In [1]:
!pip install pandas numpy matplotlib scikit-learn statsmodels prophet xgboost tensorflow flask

In [2]:
import pandas as pd

url = "https://docs.google.com/spreadsheets/d/1I1sFHSOZa9tdQfCahF1W71L4hPrJxkf1/edit?gid=1562810345#gid=1562810345"

# Convert to CSV
url = url.replace('/edit?gid=', '/export?format=csv&gid=')

df = pd.read_csv(url)

df.head()

,State,Date,Total,Category
0,Alabama,1/12/2019,"109,574,036",Beverages
1,Arizona,1/12/2019,"109,101,595",Beverages
2,Arkansas,1/12/2019,"58,049,432",Beverages
3,California,1/12/2019,"444,766,891",Beverages
4,Colorado,1/12/2019,"89,816,716",Beverages


In [4]:
df.rename(columns={
    'Order Date': 'date',
    'Region': 'state',
    'Sales': 'sales'
}, inplace=True)

In [8]:
df.columns = df.columns.str.strip().str.lower()

df.rename(columns={'total': 'sales'}, inplace=True)

df['date'] = df['date'].astype(str).str.strip()
df['date'] = pd.to_datetime(df['date'], format='%m/%d/%Y', errors='coerce')

df = df.sort_values(['state', 'date'])

In [10]:
df = df.groupby(['state', 'date'])['sales'].sum().reset_index()

In [15]:
df = df.reset_index()

In [16]:
df = df.groupby(['state', 'date'], as_index=False)['sales'].sum()

In [17]:
df = df.set_index('date')

df = df.groupby('state', group_keys=False).apply(lambda x: x.asfreq('D'))

df['sales'] = df['sales'].fillna(method='ffill')

/tmp/ipykernel_1739/4054665962.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('state', group_keys=False).apply(lambda x: x.asfreq('D'))
/tmp/ipykernel_1739/4054665962.py:5: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['sales'] = df['sales'].fillna(method='ffill')


In [18]:
df.index.is_unique

False

In [20]:
df = df.reset_index()

In [21]:
df.duplicated(subset=['state', 'date']).sum()

np.int64(71904)

In [22]:
print(df.columns)

Index(['date', 'state', 'sales'], dtype='object')


In [23]:
print(df.head())
print(df.columns)
print(df.index.name)

        date    state           sales
0 2019-01-12  Alabama    109,574,036 
1 2019-01-13      NaN    109,574,036 
2 2019-01-14      NaN    109,574,036 
3 2019-01-15      NaN    109,574,036 
4 2019-01-16      NaN    109,574,036 
Index(['date', 'state', 'sales'], dtype='object')
None


In [25]:
# Reset
df = df.reset_index()

# Ensure no duplicates
df = df.groupby(['state', 'date'], as_index=False)['sales'].sum()

# Continue pipeline

In [27]:
df['sales'] = df['sales'].astype(str).str.strip()        # remove spaces
df['sales'] = df['sales'].str.replace(',', '')           # remove commas
df['sales'] = df['sales'].astype(float)                  # convert to number

In [28]:
print(df['sales'].head())
print(df['sales'].dtype)

0    109574036.0
1    112189104.0
2    129106730.0
3    108083724.0
4    110932913.0
Name: sales, dtype: float64
float64


In [29]:
# Make sure sorted
df = df.sort_values(['state', 'date'])

def create_features(group):
    group['lag_1'] = group['sales'].shift(1)
    group['lag_7'] = group['sales'].shift(7)
    group['lag_30'] = group['sales'].shift(30)

    group['rolling_mean_7'] = group['sales'].rolling(7).mean()
    group['rolling_std_7'] = group['sales'].rolling(7).std()

    group['day_of_week'] = group['date'].dt.dayofweek
    group['month'] = group['date'].dt.month

    return group

df = df.groupby('state').apply(create_features).reset_index(drop=True)

/tmp/ipykernel_1739/250615478.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('state').apply(create_features).reset_index(drop=True)


In [30]:
df = df.dropna()

In [31]:
split_date = '2022-12-01'  # adjust if needed

train = df[df['date'] < split_date]
test = df[df['date'] >= split_date]

In [32]:
from xgboost import XGBRegressor

features = [
    'lag_1','lag_7','lag_30',
    'rolling_mean_7','rolling_std_7',
    'day_of_week','month'
]

model_xgb = XGBRegressor(n_estimators=100)
model_xgb.fit(train[features], train['sales'])

pred_xgb = model_xgb.predict(test[features])

In [33]:
from statsmodels.tsa.arima.model import ARIMA

state_df = df[df['state'] == 'California']

train_s = state_df[state_df['date'] < split_date]
test_s = state_df[state_df['date'] >= split_date]

model_arima = ARIMA(train_s['sales'], order=(5,1,0)).fit()
pred_arima = model_arima.forecast(steps=len(test_s))

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be give

In [34]:
from prophet import Prophet

prophet_df = train[['date','sales']].rename(columns={'date':'ds','sales':'y'})

model_prophet = Prophet()
model_prophet.fit(prophet_df)

future = model_prophet.make_future_dataframe(periods=len(test))
forecast = model_prophet.predict(future)

INFO:prophet:Disabling yearly seasonality. Run prophet with yearly_seasonality=True to override this.
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


In [35]:
from sklearn.metrics import mean_squared_error
import numpy as np

rmse_xgb = np.sqrt(mean_squared_error(test['sales'], pred_xgb))

print("XGBoost RMSE:", rmse_xgb)

XGBoost RMSE: 22068651.84299568


In [36]:
future_preds = model_xgb.predict(test[features].tail(56))

In [37]:
import pickle
pickle.dump(model_xgb, open('best_model.pkl','wb'))

In [ ]:
from flask import Flask, request, jsonify
import pickle
import numpy as np

# Initialize app
app = Flask(__name__)

# Load trained model
model = pickle.load(open('best_model.pkl', 'rb'))

# Home route (just to check API)
@app.route('/')
def home():
    return "Sales Forecast API is running!"

# Prediction route
@app.route('/predict', methods=['POST'])
def predict():
    data = request.json

    # Get features from request
    features = np.array(data['features']).reshape(1, -1)

    # Make prediction
    prediction = model.predict(features)

    return jsonify({
        'prediction': float(prediction[0])
    })

# Run app
app.run(host='0.0.0.0', port=5000)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
